In [47]:
import pandas as pd
import numpy as np
import pandas_ta as pta
import datetime as dt
import math

In [48]:
# Read the OHLC data from CSV file

df = pd.read_csv('NIFTY50_INDEX_15_Min.csv')
# df = pd.read_csv('NIFTYBANK_INDEX_15_Min.csv')
# df = df[:16000] # - 15 Mins - 2017 - 2020
# df = df[21000:] # - 15 Mins - 2021 - 2023

# df = df[34325:] # - 15 Mins - Feb 2023

df.head()
# df.tail(10)
# print(len(df))

,Date,Open,High,Low,Close,Volume
0,2017-07-17 09:00:00,9908.15,9908.15,9908.15,9908.15,0
1,2017-07-17 09:15:00,9908.15,9913.90,9899.50,9911.65,0
2,2017-07-17 09:30:00,9911.95,9915.15,9894.85,9897.70,0
3,2017-07-17 09:45:00,9899.05,9916.25,9898.45,9915.35,0
4,2017-07-17 10:00:00,9915.10,9916.70,9907.15,9915.30,0


In [49]:
ticker   = 'NIFTY50'
# ticker   = 'NIFTYBANK'
type1    = 'INDEX'
interval = '15'

candles_in_day  = 25
initial_capital = 100000.
position_size   = 50
brokerage_trade = 25.0

df['Date'] = pd.to_datetime(df['Date'])
df['time'] = df['Date'].dt.time
df = df[df['time'] != dt.time(9,00,00)]
df.set_index('Date', inplace=True)
df.drop(['Volume', 'time'], inplace=True, axis=1)
# df.drop(['Volume'], inplace=True, axis=1)
df.head()
# print(len(df))

,Open,High,Low,Close
Date,,,,
2017-07-17 09:15:00,9908.15,9913.90,9899.50,9911.65
2017-07-17 09:30:00,9911.95,9915.15,9894.85,9897.70
2017-07-17 09:45:00,9899.05,9916.25,9898.45,9915.35
2017-07-17 10:00:00,9915.10,9916.70,9907.15,9915.30
2017-07-17 10:15:00,9915.45,9920.30,9911.80,9916.05


In [50]:
# Calculate the Heikin Ashi (HA) candles

df['HA_Close'] = (df['Open'] + df['High'] + df['Low'] + df['Close']) / 4
df['HA_Open'] = (df['Open'] + df['Close']) / 2.0

for i in range(1, len(df)):
    df['HA_Open'][i] = (df.HA_Open[i-1] + df.HA_Close[i-1]) / 2.0

df['HA_High'] = df[['High', 'HA_Open', 'HA_Close']].max(axis=1)
df['HA_Low'] = df[['Low', 'HA_Open', 'HA_Close']].min(axis=1)

df.dropna(axis=0, inplace=True)
df.head(10)

,Open,High,Low,Close,HA_Close,HA_Open,HA_High,HA_Low
Date,,,,,,,,
2017-07-17 09:15:00,9908.15,9913.90,9899.50,9911.65,9908.3000,9909.900000,9913.900000,9899.500000
2017-07-17 09:30:00,9911.95,9915.15,9894.85,9897.70,9904.9125,9909.100000,9915.150000,9894.850000
2017-07-17 09:45:00,9899.05,9916.25,9898.45,9915.35,9907.2750,9907.006250,9916.250000,9898.450000
2017-07-17 10:00:00,9915.10,9916.70,9907.15,9915.30,9913.5625,9907.140625,9916.700000,9907.140625
2017-07-17 10:15:00,9915.45,9920.30,9911.80,9916.05,9915.9000,9910.351562,9920.300000,9910.351562
2017-07-17 10:30:00,9915.95,9920.05,9914.00,9917.75,9916.9375,9913.125781,9920.050000,9913.125781
2017-07-17 10:45:00,9917.40,9920.10,9914.40,9918.80,9917.6750,9915.031641,9920.100000,9914.400000
2017-07-17 11:00:00,9918.70,9920.15,9913.85,9915.25,9916.9875,9916.353320,9920.150000,9913.850000
2017-07-17 11:15:00,9915.40,9916.55,9908.95,9909.65,9912.6375,9916.670410,9916.670410,9908.950000


In [51]:
# n = 6

df['ST1'] = pta.supertrend(df['HA_High'], df['HA_Low'], df['HA_Close'], length=12, multiplier=2)['SUPERT_12_2.0']
df['ST2'] = pta.supertrend(df['HA_High'], df['HA_Low'], df['HA_Close'], length=24, multiplier=4)['SUPERT_24_4.0']
# df['ST3'] = pta.supertrend(df['HA_High'], df['HA_Low'], df['HA_Close'], length=30, multiplier=1)['SUPERT_30_1.0']

# df['ST3'] = pta.supertrend(df['HA_High'], df['HA_Low'], df['HA_Close'], length=10, multiplier=5)['SUPERT_10_5.0']

# df['ST1_HT'] = pta.supertrend(df['HA_High'].rolling(n).mean(), 
#                               df['HA_Low'].rolling(n).mean(), 
#                               df['HA_Close'].rolling(n).mean(), length=21, multiplier=1)['SUPERT_21_1.0']

# df['ST2_HT'] = pta.supertrend(df['HA_High'].rolling(n).mean(), 
#                               df['HA_Low'].rolling(n).mean(), 
#                               df['HA_Close'].rolling(n).mean(), length=14, multiplier=2)['SUPERT_14_2.0']

# df['ST3_HT'] = pta.supertrend(df['HA_High'].rolling(n).mean(), 
#                               df['HA_Low'].rolling(n).mean(), 
#                               df['HA_Close'].rolling(n).mean(), length=7, multiplier=3)['SUPERT_7_3.0']

df = df.iloc[1:, :]
df.dropna(axis=0, inplace=True)
df.head()

,Open,High,Low,Close,HA_Close,HA_Open,HA_High,HA_Low,ST1,ST2
Date,,,,,,,,,,
2017-07-17 15:15:00,9919.05,9921.30,9910.40,9913.65,9916.1000,9921.306267,9921.306267,9910.40,9906.189863,9880.304424
2017-07-18 09:15:00,9863.50,9871.55,9853.25,9871.55,9864.9625,9918.703133,9918.703133,9853.25,9913.218246,9930.952936
2017-07-18 09:30:00,9870.30,9877.65,9866.40,9871.70,9871.5125,9891.832817,9891.832817,9866.40,9908.566334,9926.697148
2017-07-18 09:45:00,9872.35,9884.45,9872.35,9876.40,9876.3875,9881.672658,9884.450000,9872.35,9907.552015,9926.154459
2017-07-18 10:00:00,9877.25,9877.25,9873.35,9873.35,9875.3000,9879.030079,9879.030079,9873.35,9903.859400,9922.901409


In [52]:
# Generate signals for Buy and Sell

def generate_signals():
    
    buy_signal = False
    sell_signal = False    

    for i in range(len(df)):

        if not buy_signal and df['HA_Close'][i-1] > df['ST1'][i-1] \
                          and df['HA_Close'][i-1] > df['ST2'][i-1]:
                        #   and df['HA_Close'][i-1] > df['ST3'][i-1]:

            buy_signal = True
            sell_signal = False
            df['Signal'][i] = 'Buy'

        if not sell_signal and df['HA_Close'][i-1] < df['ST1'][i-1] \
                           and df['HA_Close'][i-1] < df['ST2'][i-1]:
                        #    and df['HA_Close'][i-1] < df['ST3'][i-1]:

            sell_signal = True
            buy_signal = False
            df['Signal'][i] = 'Sell'

In [53]:
# # Generate signals for Buy and Sell
# def generate_signals(df):
#     buy_signal = False
#     sell_signal = False
#     signals = []
#     for i in range(len(df)):
#         #   and df['HA_Close'][i-1] > df['ST3'][i-1]:
#         if not buy_signal and df['HA_Close'][i-1] > df['ST1'][i-1] \
#                           and df['HA_Close'][i-1] > df['ST2'][i-1]:# \
#                         #   and df['HA_Close'][i-1] > df['ST2_HT'][i-1]:

#         # if not buy_signal and df['HA_Close'][i-1] > df['ST1_HT'][i-1] \
#         #                   and df['HA_Close'][i-1] > df['ST2_HT'][i-1] \
#         #                   and df['HA_Close'][i-1] > df['ST3_HT'][i-1]:

#             buy_signal = True
#             signals.append('Buy')

#         elif buy_signal and df['HA_Close'][i-1] < df['ST1'][i-1]:
#                         # and df['HA_Close'][i-1] < df['ST2'][i-1]:
        
#         # elif buy_signal and df['HA_Close'][i-1] < df['ST1_HT'][i-1] \
#         #                 and df['HA_Close'][i-1] < df['ST2_HT'][i-1]:

#             buy_signal = False
#             signals.append('Close Buy')
#         # and df['HA_Close'][i-1] < df['ST3'][i-1]:
#         elif not sell_signal and df['HA_Close'][i-1] < df['ST1'][i-1] \
#                              and df['HA_Close'][i-1] < df['ST2'][i-1]: #\
#                             #  and df['HA_Close'][i-1] < df['ST2_HT'][i-1]:

#         # elif not sell_signal and df['HA_Close'][i-1] < df['ST1_HT'][i-1] \
#         #                     and df['HA_Close'][i-1] < df['ST2_HT'][i-1] \
#         #                     and df['HA_Close'][i-1] < df['ST3_HT'][i-1]:

#             sell_signal = True
#             signals.append('Sell')

#         elif sell_signal and df['HA_Close'][i-1] > df['ST1'][i-1]:# \
#                         #  and df['HA_Close'][i-1] > df['ST2'][i-1]:
        
#         # elif sell_signal and df['HA_Close'][i-1] > df['ST1_HT'][i-1] \
#         #                  and df['HA_Close'][i-1] > df['ST2_HT'][i-1]:

#             sell_signal = False
#             signals.append('Close Sell')

#         else:
#             signals.append('Hold')
            
    # return signals

In [54]:
df['Signal'] = 'Hold'

generate_signals()

df.dropna(axis=0, inplace=True)
df.head()

C:\Users\Sushant.Dhumak\AppData\Local\Temp\ipykernel_11460\2885500971.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Signal'][i] = 'Sell'
C:\Users\Sushant.Dhumak\AppData\Local\Temp\ipykernel_11460\2885500971.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Signal'][i] = 'Buy'


,Open,High,Low,Close,HA_Close,HA_Open,HA_High,HA_Low,ST1,ST2,Signal
Date,,,,,,,,,,,
2017-07-17 15:15:00,9919.05,9921.30,9910.40,9913.65,9916.1000,9921.306267,9921.306267,9910.40,9906.189863,9880.304424,Sell
2017-07-18 09:15:00,9863.50,9871.55,9853.25,9871.55,9864.9625,9918.703133,9918.703133,9853.25,9913.218246,9930.952936,Buy
2017-07-18 09:30:00,9870.30,9877.65,9866.40,9871.70,9871.5125,9891.832817,9891.832817,9866.40,9908.566334,9926.697148,Sell
2017-07-18 09:45:00,9872.35,9884.45,9872.35,9876.40,9876.3875,9881.672658,9884.450000,9872.35,9907.552015,9926.154459,Hold
2017-07-18 10:00:00,9877.25,9877.25,9873.35,9873.35,9875.3000,9879.030079,9879.030079,9873.35,9903.859400,9922.901409,Hold


In [55]:
# df.to_csv('TrippleSuperTrend_Signal1' + '.csv')

In [56]:
# Backtest the strategy
def backtest(df):
    total_profit_loss = initial_capital
    buy_value = 0.
    buy_sell_value = 0.
    sell_value = 0.
    sell_buy_value = 0.
    profit_loss = 0.    
    slippage = 5.0
    trade_count = 0
    Buy_ON = False
    Sell_ON = False
    start_dt1 = ''
    start_dt2 = ''
    end_dt = ''
    flag1 = False
    flag2 = False
    max_loss = 100000
    max_profit = -100000
    max_buy_count = 0
    max_sell_count = 0
    max_buy = 0
    max_sell = 0
    # start_dt_buy = ''
    # end_dt_buy = ''
    # start_dt_sell = ''
    # end_dt_sell = ''

    #
    #  stoploss = 80.0

    trade_df = pd.DataFrame(columns=['Signal','DateTime','Price','Profit','Cum Profit'])

    convert_dict = {'Price': float,
                    'Profit': float,
                    'Cum Profit': float,
                    }
 
    trade_df = trade_df.astype(convert_dict)

    j = 0
    
    for i in range(len(df)):        

        # if i == 0 and flag1 == False:
        #     start_dt = str(df.index[i])
        #     flag1 = True
                    
        # if i == len(df)-1 and flag2 == False:
        #     end_dt = str(df.index[i])
        #     flag2 = True

        signal = df.iloc[i, -1]

        if signal == 'Buy' and Buy_ON == False:
            buy_value = df.iloc[i, 0] + slippage
            trade_count += 1
            Buy_ON = True
            max_buy_count = 1

            trade_df.loc[j] = (['Buy', str(df.index[i]), str(buy_value), '', ''])

            if Sell_ON == True:
                sell_buy_value = df.iloc[i, 0] + slippage
                profit_loss = (sell_value - sell_buy_value) * position_size
                total_profit_loss += profit_loss
                trade_count += 1
                Sell_ON = False
                j += 1

                if max_sell < max_sell_count:
                    max_sell = max_sell_count                
                    max_sell_count = 0

                trade_df.loc[j] = (['Closed Sell', str(df.index[i]), str(sell_buy_value), str(profit_loss), str(total_profit_loss)])

        if signal == 'Sell' and Sell_ON == False:
            sell_value = df.iloc[i, 0] - slippage
            trade_count += 1
            Sell_ON = True
            max_sell_count = 1

            trade_df.loc[j] = (['Sell', str(df.index[i]), str(sell_value), '', ''])
        
            if Buy_ON == True:
                buy_sell_value = df.iloc[i, 0] - slippage
                profit_loss = (buy_sell_value - buy_value) * position_size
                total_profit_loss += profit_loss
                trade_count += 1
                Buy_ON = False
                j += 1

                if max_buy < max_buy_count:
                    max_buy = max_buy_count
                    max_buy_count = 0

                trade_df.loc[j] = (['Close Buy', str(df.index[i]), str(buy_sell_value), str(profit_loss), str(total_profit_loss)])

        j += 1

        if profit_loss > max_profit:
            max_profit = profit_loss

        if profit_loss < max_loss:
            max_loss = profit_loss

        max_buy_count += 1
        max_sell_count += 1
    
    # print(start_dt_buy)
    # print(start_dt_sell)

    # print(end_dt_buy)
    # print(end_dt_sell)

    # print(max_buy)
    # print(max_sell)

    return total_profit_loss, trade_count, max_profit, max_loss, trade_df, max_buy, max_sell

In [57]:
def MDD(trade_df):
    
    DD_df = pd.DataFrame()
    DD_df = trade_df.copy()
    DD_df.replace('', np.nan, inplace=True)
    DD_df = DD_df.dropna()

    max_values = DD_df['Cum Profit'].rolling(window=len(DD_df), min_periods = 1).max()
    DD_values =  DD_df['Cum Profit'].astype(float) / max_values - 1
    MDD_values = DD_values.rolling(window=len(DD_df), min_periods = 1).min()
    DD = MDD_values.min() * 100    

    return DD

In [58]:
profit_loss, trade_count, max_profit, max_loss, trade_df, max_buydays, max_selldays = backtest(df)

print('Initial Capital  :', round(initial_capital, 2))
print('Total Profit/Loss:', round(profit_loss, 2))
print('Number of Trades :', trade_count)
print('Brokerage        :', (trade_count * brokerage_trade))
print('Net Profit/Loss  :', round((profit_loss - (trade_count * brokerage_trade)), 2))
print('Profit/Loss %    :', str(round((((profit_loss - (trade_count * brokerage_trade) - initial_capital) / initial_capital) * 100.), 2)) + '%')
print('Max Profit       :', round((max_profit - brokerage_trade), 2))
print('Max Loss         :', round((max_loss - brokerage_trade), 2))
print('Max Drawdown     :', str(round(MDD(trade_df), 2)) + '%')
print('Max Buy Days     :', str(math.ceil(max_buydays/candles_in_day)) + ' Days')
print('Max Sell Days    :', str(math.ceil(max_selldays/candles_in_day)) + ' Days')

Initial Capital  : 100000.0
Total Profit/Loss: 412322.5
Number of Trades : 977
Brokerage        : 24425.0
Net Profit/Loss  : 387897.5
Profit/Loss %    : 287.9%
Max Profit       : 74400.0
Max Loss         : -29260.0
Max Drawdown     : -50.63%
Max Buy Days     : 13 Days
Max Sell Days    : 12 Days


In [59]:
trade_df.reset_index(inplace=True)
trade_df.to_csv('TrippleSuperTrend_2023'+ '_' + ticker + '_' + type1 + '_' + interval + '_Min' + '.csv')

In [60]:
# Initial Capital  : 100000.0
# Total Profit/Loss: 482770.0
# Number of Trades : 713
# Brokerage        : 17825.0
# Net Profit/Loss  : 464945.0
# Profit/Loss %    : 364.95%
# Max Profit       : 108035.0
# Max Loss         : -42570.0
# Max Drawdown     : -41.67%
# Max Buy Days     : 17 Days
# Max Sell Days    : 17 Days